# M9.6 · Kenar tespiti — Kirsch, Marr-Hildreth, Canny

Kenar modelleri (basamak, rampa, çatı), gürültünün türevleri nasıl bozduğu, gradyan operatörleri ve pusula (Kirsch) çekirdekleri; Laplacian of Gaussian ile sıfır geçişleri (Marr-Hildreth) ve Canny'nin dört adımı. Elle Sobel, yön nicemleme, çekirdek boyutu ve histerezis; saf Python'da non-maxima suppression, histerezis ve sıfır geçişleri; NumPy'da `cv2.Canny` ile piksel piksel aynı Canny; PyTorch'ta gürültülü basamaklardan öğrenilen 1-B filtrenin Gauss türevine dönüştüğünü (Canny'nin optimallik sonucu) görüyor; en sonda tüm toplu işi GPU'ya uygun tensör işlemleriyle yapan ve yine `cv2.Canny` ile birebir aynı sonuç veren bir Canny yazıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m9-6-edges.html](https://m-gonul.github.io/pytorch-lab/m9-6-edges.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m9_6_edges.py
CHECKS = {
 "b": "import numpy as np\nimport cv2\nfrom scipy import ndimage\nfrom skimage import data\nfrom labcheck import no_imports, need, ensure\n\n\ndef ref_nms(m, gx, gy):\n    H, W = m.shape\n    P = np.pad(m, 1)\n    a = np.degrees(np.arctan2(gy, gx)) % 180\n    sh = lambda dy, dx: P[1 + dy:1 + dy + H, 1 + dx:1 + dx + W]\n    d0 = (a < 22.5) | (a >= 157.5); d1 = (a >= 22.5) & (a < 67.5); d2 = (a >= 67.5) & (a < 112.5); d3 = (a >= 112.5) & (a < 157.5)\n    n1 = np.select([d0, d1, d2, d3], [sh(0, -1), sh(-1, -1), sh(-1, 0), sh(-1, 1)])\n    n2 = np.select([d0, d1, d2, d3], [sh(0, 1), sh(1, 1), sh(1, 0), sh(1, -1)])\n    return np.where((m < n1) | (m < n2), 0, m)\n\n\ndef ref_hys(g, lo, hi):\n    cand = g >= lo\n    lab, n = ndimage.label(cand, np.ones((3, 3)))\n    ok = np.zeros(n + 1, bool); ok[np.unique(lab[g >= hi])] = True; ok[0] = False\n    return ok[lab].astype(int)\n\n\ndef ref_zc(g, thr):\n    H, W = g.shape\n    out = np.zeros((H, W), int)\n    c = lambda dy, dx: g[1 + dy:H - 1 + dy, 1 + dx:W - 1 + dx]\n    z = np.zeros((H - 2, W - 2), bool)\n    for (a, b), (cc, d) in [((0, -1), (0, 1)), ((-1, 0), (1, 0)), ((-1, -1), (1, 1)), ((-1, 1), (1, -1))]:\n        p, n = c(a, b), c(cc, d)\n        z |= (p * n < 0) & (np.abs(p - n) > thr)\n    out[1:-1, 1:-1] = z\n    return out\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/cv2 kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"cv2\", \"scipy\", \"skimage\"}))\n    img = cv2.GaussianBlur(data.camera()[180:230, 230:290], (5, 5), 1.2).astype(np.float64)\n    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0); gy = cv2.Sobel(img, cv2.CV_64F, 0, 1); m = np.hypot(gx, gy)\n\n    def t_nms():\n        got = np.array(need(ns, \"nms\")(m.tolist(), gx.tolist(), gy.tolist()))\n        ensure(np.allclose(got, ref_nms(m, gx, gy)), \"başvuru NMS ile farklı\")\n        return f\"{(m > 0).sum()} → {(got > 0).sum()} sıfır olmayan piksel\"\n    R.test(\"nms\", t_nms)\n\n    def t_hys():\n        n = ref_nms(m, gx, gy)\n        lo, hi = np.percentile(n[n > 0], [50, 85])\n        got = np.array(need(ns, \"hysteresis\")(n.tolist(), lo, hi))\n        ensure(np.array_equal(got, ref_hys(n, lo, hi)), \"başvuru (bağlı bileşenler) ile farklı\")\n        ensure(ns[\"hysteresis\"]([[0, 40, 120, 60, 0, 70, 0]], 50, 100) == [[0, 0, 1, 1, 0, 0, 0]], \"elle örnek\")\n    R.test(\"hysteresis\", t_hys)\n\n    def t_zc():\n        g = cv2.Laplacian(cv2.GaussianBlur(data.camera()[100:150, 100:160].astype(np.float64), (9, 9), 1.4), cv2.CV_64F)\n        for thr in (0.0, 0.04 * g.max()):\n            got = np.array(need(ns, \"zero_crossings\")(g.tolist(), thr))\n            ensure(np.array_equal(got, ref_zc(g, thr)), f\"eşik {thr:.2f}\")\n    R.test(\"zero_crossings\", t_zc)\n",
 "c": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import no_loops, need, ensure\n\n\ndef ref_mh(img, sigma, thr_frac):\n    n = int(np.ceil(6 * sigma)); n += 1 - n % 2\n    g = cv2.Laplacian(cv2.GaussianBlur(img.astype(np.float64), (n, n), sigma), cv2.CV_64F)\n    thr = thr_frac * g.max(); H, W = g.shape\n    z = np.zeros((H, W), bool)\n    c = lambda a, b: g[1 + a:H - 1 + a, 1 + b:W - 1 + b]\n    for p, q in [(c(0, -1), c(0, 1)), (c(-1, 0), c(1, 0)), (c(-1, -1), c(1, 1)), (c(-1, 1), c(1, -1))]:\n        z[1:-1, 1:-1] |= (p * q < 0) & (np.abs(p - q) > thr)\n    return z\n\n\ndef gray(name):\n    im = getattr(data, name)()\n    return cv2.cvtColor(im, cv2.COLOR_RGB2GRAY) if im.ndim == 3 else im\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    rows = []\n    for name, lo, hi, l2 in [(\"camera\", 50, 150, False), (\"camera\", 30, 90, True), (\"coins\", 100, 200, False), (\"astronaut\", 20, 60, True), (\"moon\", 40, 100.7, False)]:\n        def t(name=name, lo=lo, hi=hi, l2=l2):\n            img = gray(name)\n            got = need(ns, \"canny_cv\")(img, lo, hi, l2)\n            want = cv2.Canny(img, lo, hi, L2gradient=l2)\n            diff = int((got != want).sum())\n            ensure(diff == 0, f\"{diff} piksel farklı (toplam kenar {int((want > 0).sum())})\")\n            rows.append([name, f\"{lo}/{hi}\", \"L2\" if l2 else \"L1\", int((want > 0).sum())])\n        R.test(f\"canny_cv = cv2.Canny ({name}, {lo}/{hi}, {'L2' if l2 else 'L1'})\", t)\n    if rows:\n        R.table(\"Birebir eşleşen Canny çıktıları\", [\"görüntü\", \"eşikler\", \"norm\", \"kenar pikseli\"], rows)\n\n    def mh():\n        img = gray(\"camera\")[::2, ::2]\n        for s, f in [(1.4, 0.0), (2.0, 0.04), (3.0, 0.04)]:\n            got = need(ns, \"marr_hildreth\")(img, s, f)\n            ensure(got.dtype == bool and np.array_equal(got, ref_mh(img, s, f)), f\"σ = {s}, eşik %{int(f * 100)}\")\n        return f\"σ = 2: eşiksiz {ref_mh(img, 2.0, 0).sum()} piksel, %4 eşikle {ns['marr_hildreth'](img, 2.0, 0.04).sum()} piksel (eşik 'spagetti' kenarları temizler)\"\n    R.test(\"marr_hildreth\", mh)\n",
 "d": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def dk():\n        k = need(ns, \"dog_kernel\")(1.5, 9)\n        x = np.arange(9) - 4\n        close(np.asarray(k, dtype=np.float64), x / 2.25 * np.exp(-x ** 2 / 4.5), atol=1e-6)\n    R.test(\"dog_kernel\", dk)\n\n    def learn():\n        w = need(ns, \"learn_edge_filter\")().double()\n        ensure(w.shape == (21,), \"21 katsayı\")\n        best = max((float(np.corrcoef(w.numpy(), ns[\"dog_kernel\"](s, 21).double().numpy())[0, 1]), s) for s in np.arange(0.5, 6.01, 0.1))\n        anti = float(np.corrcoef(w.numpy(), -w.flip(0).numpy())[0, 1])\n        ensure(best[0] > 0.95, f\"Gauss türeviyle en iyi korelasyon {best[0]:.3f} (σ = {best[1]:.1f}); 0.95 üstü bekleniyordu\")\n        ensure(anti > 0.9, f\"tek (antisimetrik) değil: {anti:.3f}\")\n        R.table(\"Öğrenilen filtre\", [\"ölçü\", \"değer\"], [[\"Gauss türeviyle korelasyon\", f\"{best[0]:.3f}\"], [\"en iyi σ\", f\"{best[1]:.1f}\"], [\"antisimetri\", f\"{anti:.3f}\"]],\n                note=\"Kimse 'Gauss türevi' demedi: gürültülü basamakları bulmayı öğrenen doğrusal filtre kendiliğinden ona yakınsıyor (Canny 1986).\")\n    R.test(\"öğrenilen filtre ≈ Gauss türevi\", learn)\n\n    def sob():\n        img = data.camera()[100:180, 50:150].astype(np.float64)\n        gx, gy = need(ns, \"sobel_torch\")(torch.tensor(img))\n        close(gx.numpy(), cv2.Sobel(img, cv2.CV_64F, 1, 0, borderType=cv2.BORDER_REPLICATE), atol=1e-9, what=\"gx\")\n        close(gy.numpy(), cv2.Sobel(img, cv2.CV_64F, 0, 1, borderType=cv2.BORDER_REPLICATE), atol=1e-9, what=\"gy\")\n        ensure(ns[\"__source__\"].count(\"conv2d\") == 1, \"tek conv2d çağrısı\")\n    R.test(\"sobel_torch = cv2.Sobel\", sob)\n",
 "e": "import numpy as np\nimport cv2\nimport torch\nfrom skimage import data\nfrom labcheck import max_loops, need, ensure, bench, fmt_time\n\n\ndef gray(name):\n    im = getattr(data, name)()\n    return cv2.cvtColor(im, cv2.COLOR_RGB2GRAY) if im.ndim == 3 else im\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"tek döngü (histerezis)\", lambda: max_loops(src, 1, func=\"canny_torch\"))\n    crops = np.stack([gray(n)[40:200, 60:260] for n in (\"camera\", \"coins\", \"astronaut\", \"moon\")])\n    for lo, hi, l2 in [(50, 150, False), (30, 90, True), (20, 60.5, False)]:\n        def t(lo=lo, hi=hi, l2=l2):\n            got = need(ns, \"canny_torch\")(torch.tensor(crops), lo, hi, l2).numpy()\n            for i in range(len(crops)):\n                want = cv2.Canny(crops[i], lo, hi, L2gradient=l2)\n                ensure(np.array_equal(got[i], want), f\"görüntü {i}: {(got[i] != want).sum()} piksel farklı\")\n        R.test(f\"canny_torch = cv2.Canny ({lo}/{hi}, {'L2' if l2 else 'L1'})\", t)\n    if \"canny_torch\" not in ns:\n        return\n    big = np.stack([cv2.resize(gray(n), (256, 256)) for n in (\"camera\", \"coins\", \"astronaut\", \"moon\")] * 2)\n    tb = torch.tensor(big)\n    t_cv = bench(lambda: [cv2.Canny(b, 50, 150) for b in big], repeat=3)\n    t_t = bench(lambda: ns[\"canny_torch\"](tb, 50, 150), repeat=3)\n    R.table(\"8 × 256 × 256\", [\"yöntem\", \"süre\"], [[\"cv2.Canny döngüsü\", fmt_time(t_cv)], [\"canny_torch (toplu)\", fmt_time(t_t)]],\n            note=\"CPU'da OpenCV'nin elle optimize edilmiş C++'ı kazanır; histerezisin yinelemeli genişlemesi en uzun zincir kadar sürer. GPU'da ve büyük yığınlarda tablo değişir.\")\n",
 "x1": "import numpy as np\nimport cv2\nfrom skimage import data\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    ring = [5, 5, 5, -3, -3, -3, -3, -3]\n    pos = [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (1, 0)]\n    ks = []\n    for i in range(8):\n        k = np.zeros((3, 3))\n        for (y, x), v in zip(pos, np.roll(ring, -i)):\n            k[y, x] = v\n        ks.append(k)\n\n    def exact():\n        img = data.camera()[200:300, 100:220]\n        Rr = np.stack([cv2.filter2D(img.astype(np.float64), -1, k, borderType=cv2.BORDER_REPLICATE) for k in ks])\n        r, i = need(ns, \"kirsch\")(img)\n        close(r, Rr.max(0), atol=1e-9, what=\"yanıt\")\n        ensure(np.array_equal(i, Rr.argmax(0)), \"yön indisi\")\n    R.test(\"kirsch = 8 çekirdeğin en büyüğü\", exact)\n\n    def dirs():\n        names = [\"K\", \"KB\", \"B\", \"GB\", \"G\", \"GD\", \"D\", \"KD\"]\n        rows = []\n        for i, (dy, dx) in enumerate([(-1, 0), (-1, -1), (0, -1), (1, -1), (1, 0), (1, 1), (0, 1), (-1, 1)]):\n            yy, xx = np.mgrid[-20:21, -20:21]\n            img = ((yy * dy + xx * dx) > 0).astype(np.uint8) * 200\n            r, idx = ns[\"kirsch\"](img)\n            got = int(idx[20, 20])\n            ensure(got == i, f\"parlak taraf {names[i]} yönündeyken indis {got}\")\n            rows.append([names[i], got, int(r[20, 20])])\n        R.table(\"Basamak yönü → Kirsch indisi\", [\"parlak taraf\", \"indis\", \"yanıt\"], rows)\n    R.test(\"8 yön doğru\", dirs)\n",
 "x2": "import numpy as np\nimport cv2\nfrom skimage import data, feature\nfrom skimage.transform import hough_line\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    th = np.linspace(-np.pi / 2, np.pi / 2, 180, endpoint=False)\n\n    def same():\n        e = feature.canny(data.camera()[::2, ::2].astype(float), 2)\n        acc, d = need(ns, \"hough_lines\")(e, th)\n        A, _, D = hough_line(e, th)\n        ensure(acc.shape == A.shape and np.array_equal(acc, A), \"birikim dizisi skimage ile farklı\")\n        close(d, D, what=\"dists\")\n    R.test(\"hough_lines = skimage.transform.hough_line\", same)\n\n    def find():\n        img = np.zeros((200, 240), np.uint8)\n        truth = [(60.0, np.deg2rad(-30)), (100.0, np.deg2rad(45)), (120.0, np.deg2rad(80))]\n        for rho, t in truth:\n            c, s = np.cos(t), np.sin(t)\n            x0, y0 = rho * c, rho * s\n            p1 = (int(round(x0 - 1000 * s)), int(round(y0 + 1000 * c))); p2 = (int(round(x0 + 1000 * s)), int(round(y0 - 1000 * c)))\n            cv2.line(img, p1, p2, 255, 1)\n        rng = np.random.default_rng(0)\n        img[rng.random(img.shape) < 0.01] = 255\n        acc, d = ns[\"hough_lines\"](img > 0, th)\n        got = need(ns, \"top_lines\")(acc, th, d, 3)\n        rows = []\n        for rho, t in truth:\n            best = min(got, key=lambda g: abs(g[0] - rho) + 50 * abs(g[1] - t))\n            ensure(abs(best[0] - rho) <= 2 and abs(np.rad2deg(best[1] - t)) <= 2, f\"(ρ = {rho}, θ = {np.rad2deg(t):.0f}°) bulunamadı\")\n            rows.append([f\"{rho:.0f}, {np.rad2deg(t):.0f}°\", f\"{best[0]:.0f}, {np.rad2deg(best[1]):.0f}°\"])\n        R.table(\"%1 tuz gürültüsü altında üç doğru\", [\"gerçek (ρ, θ)\", \"bulunan\"], rows)\n    R.test(\"top_lines doğruları buluyor\", find)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    def size():\n        for s, n in [(0.5, 3), (1.0, 7), (1.4, 9), (2.0, 13), (2.5, 15)]:\n            ensure(need(ns, \"gaussian_size\")(s) == n, f\"σ = {s}: {n} bekleniyordu\")\n    R.test(\"gaussian_size\", size)\n\n    def kern():\n        rows = []\n        for s in (1.0, 2.0, 3.0):\n            a = need(ns, \"log_formula\")(s); b = need(ns, \"log_via_laplacian\")(s)\n            n = ns[\"gaussian_size\"](s)\n            ensure(a.shape == (n, n) and b.shape == (n + 2, n + 2), \"boyutlar\")\n            ensure(abs(a.sum()) < 1e-9 and abs(b.sum()) < 1e-12, \"toplam sıfır\")\n            c = float(np.corrcoef(a.ravel(), b[1:-1, 1:-1].ravel())[0, 1])\n            ensure(c > 0.97, f\"σ = {s}: iki çekirdeğin korelasyonu {c:.3f}\")\n            mid = a[n // 2]\n            zc = np.nonzero(np.diff(np.sign(mid[n // 2:])))[0]\n            rows.append([s, n, f\"{c:.4f}\", f\"{zc[0] + 0.5 if len(zc) else '—'}\", f\"{np.sqrt(2) * s:.2f}\"])\n        R.table(\"LoG çekirdekleri\", [\"σ\", \"n\", \"korelasyon\", \"ilk işaret değişimi (piksel)\", \"√2σ\"], rows)\n    R.test(\"iki LoG yolu aynı şekli verir\", kern)\n\n    def noise():\n        rows = []\n        for v in (0.1, 1.0, 10.0):\n            d1, d2 = need(ns, \"derivative_noise\")(v)\n            s = np.sqrt(v)\n            ensure(abs(d1 / (np.sqrt(2) * s) - 1) < 0.25 and abs(d2 / (np.sqrt(6) * s) - 1) < 0.25, f\"σ² = {v}: türev gürültüsü √2σ ve √6σ civarında olmalı\")\n            e1, e2 = ns[\"derivative_noise\"](v, smooth=2.0)\n            ensure(e1 < d1 / 3 and e2 < d2 / 5, \"yumuşatma türev gürültüsünü azaltmalı\")\n            rows.append([v, f\"{s:.3f}\", f\"{d1:.3f}\", f\"{d2:.3f}\", f\"{e1:.4f}\", f\"{e2:.4f}\"])\n        R.table(\"Düz bölgede türev gürültüsü (rampa yüksekliği 200, eğim 3.1/piksel)\", [\"σ²\", \"σ\", \"1. fark\", \"2. fark\", \"1. fark (Gauss σ=2)\", \"2. fark (Gauss σ=2)\"], rows,\n                note=\"İkinci farkın gürültüsü √6σ: σ² = 10'da bile 7.7, yani rampanın kendi ikinci türevini (yalnız iki uçta ≈ 3) gömer.\")\n    R.test(\"gürültü türevleri\", noise)\n"
}

In [ ]:
import importlib, labcheck, checks_m9_6_edges as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Kenar modelleri:** *basamak* (bir pikselde geçiş), *rampa* (lens ya da sıkıştırma bulanıklığıyla yayılmış basamak), *çatı* (ince çizgi). Bir rampada birinci türev rampa boyunca pozitif, düz bölgede sıfırdır; ikinci türev rampanın başında pozitif, sonunda negatiftir; ikisinin arasındaki **sıfır geçişi** kalın kenarın merkezini verir.

**Gürültü** türevlerle büyür: $\sigma^2=0.1$ kadar küçük bir Gauss gürültüsü ikinci türevi tanınmaz hale getirir. Bu yüzden her dedektör üç adımdır: **yumuşat → aday noktaları bul → yerelleştir.**

**Gradyan:** $\nabla f=(g_x,g_y)$, büyüklük $M=\sqrt{g_x^2+g_y^2}$ (ya da $|g_x|+|g_y|$), yön $\alpha=\operatorname{atan2}(g_y,g_x)$. Kenarın kendisi gradyana diktir. Operatörler: Roberts (2 × 2), Prewitt, Sobel (merkezde 2 ağırlığıyla ek yumuşatma), **Kirsch** (8 pusula yönü, en güçlü yanıt seçilir).

**Marr-Hildreth:** $g=\nabla^2G*f$, ardından sıfır geçişleri. LoG $\nabla^2G=\frac{x^2+y^2-2\sigma^2}{\sigma^4}e^{-\frac{x^2+y^2}{2\sigma^2}}$ ("Meksika şapkası"), sıfırları $r=\sqrt2\sigma$ çemberinde. Çekirdek boyutu $n$: $\lceil6\sigma\rceil$'dan büyük ya da eşit en küçük tek sayı. Sıfır geçişi: $3\times3$ komşulukta dört karşıt çiftten (sol/sağ, üst/alt, iki çapraz) en az birinin işaretleri farklı **ve** farkları eşikten büyük.

**Canny** (1986; düşük hata, iyi yerelleşme, tek yanıt): 1-B gürültülü basamak için optimal dedektörün iyi bir yaklaşığı **Gauss'un türevidir**. 2-B'de:

1. Gauss ile yumuşat.
2. Gradyan büyüklüğü ve açısı.
3. non-maxima suppression: açıyı 4 yöne nicemle (0°, 45°, 90°, 135°; ±22.5° aralıklar), gradyan yönündeki iki komşudan küçükse sıfırla.
4. histerezis eşikleme: $T_H$ üstü güçlü, $[T_L,T_H)$ zayıf; güçlü bir piksele 8-komşulukla bağlı zayıflar kalır. Canny $T_H:T_L$ = 2:1 ile 3:1 önerir.

## 1 · Elle

Açı nicemleme formülü: $q=\big\lfloor((\alpha\bmod180)+22.5)/45\big\rfloor\bmod4$, yön $45q$.

Histerezis satırında 60 zayıf ama 120'ye komşu, kalıyor; 70 zayıf ve yalnız, gidiyor.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Sobel (OpenCV yönü: x sağa, y aşağı), yama $\begin{bmatrix}0&0&50\\0&50&100\\50&100&100\end{bmatrix}$, merkezde**

- `gx`: $g_x$
- `gy`: $g_y$
- `m2`: $M=\sqrt{g_x^2+g_y^2}$
- `m1`: $|g_x|+|g_y|$ (OpenCV varsayılanı)

**Canny yön nicemleme: normal açısını 0, 45, 90, 135'ten birine ata**

- `q100`: $\alpha=100°$
- `q160`: $\alpha=160°$
- `q30`: $\alpha=-30°$ (yani 150°)

**Ölçek**

- `n14`: $\sigma=1.4$ için Gauss çekirdek boyutu $n$ (en küçük tek $\ge\lceil6\sigma\rceil$)
- `n2`: $\sigma=2$ için $n$
- `zc`: $\sigma=2$ LoG'un sıfır geçiş çemberinin yarıçapı

**Histerezis ve Kirsch**

- `hys`: NMS sonrası satır $[0, 40, 120, 60, 0, 70, 0]$, $T_L=50$, $T_H=100$, 8-komşuluk: kaç kenar pikseli kalır?
- `kir`: Kirsch K çekirdeği $\begin{bmatrix}5&5&5\\-3&0&-3\\-3&-3&-3\end{bmatrix}$'nin üstü 100, altı 0 olan basamaktaki yanıtı

In [ ]:
cevaplar = dict(
    gx=None,
    gy=None,
    m2=None,
    m1=None,
    q100=None,
    q160=None,
    q30=None,
    n14=None,
    n2=None,
    zc=None,
    hys=None,
    kir=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"gx":[300,0],"gy":[300,0],"m2":[424.264,0.001],"m1":[600,0],"q100":[90,0],"q160":[0,0],"q30":[135,0],"n14":[9,0],"n2":[13,0],"zc":[2.828427,0.001],"hys":[2,0],"kir":[1500,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, liste listeleri; görüntü dışında kalan komşular **0** sayılır.

- `nms(mag, gx, gy)` — $\alpha$ = `degrees(atan2(gy, gx)) % 180`; $[0,22.5)\cup[157.5,180)$ → sol/sağ; $[22.5,67.5)$ → $(y-1,x-1)$ ve $(y+1,x+1)$ (y aşağı!); $[67.5,112.5)$ → üst/alt; $[112.5,157.5)$ → $(y-1,x+1)$ ve $(y+1,x-1)$. İki komşudan **herhangi birinden küçükse** 0, değilse büyüklük.
- `hysteresis(gn, lo, hi)` — güçlü $\ge$ `hi`, zayıf $[$`lo`, `hi`$)$; güçlülerden 8-komşulukla genişleyen BFS (`collections.deque`). Çıktı 0/1.
- `zero_crossings(g, thr)` — kenarda olmayan her piksel için dört karşıt çift; $g_ag_b<0$ ve $|g_a-g_b|>$ `thr` ise 1. Görüntü kenarı 0.

In [ ]:
%%writefile stage_b.py
import math
from collections import deque


def nms(mag, gx, gy):
    ...


def hysteresis(gn, lo, hi):
    ...


def zero_crossings(g, thr):
    ...


print(hysteresis([[0, 40, 120, 60, 0, 70, 0]], 50, 100))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, **döngü yok**; `cv2.Sobel` ve `scipy.ndimage.label` serbest.

**`canny_cv(img, lo, hi, l2=False)`** — `cv2.Canny(img, lo, hi, L2gradient=l2)` ile **piksel piksel aynı**. OpenCV'nin ayrıntıları (`canny.cpp`):

1. $d_x, d_y$ = `cv2.Sobel(img, cv2.CV_16S, …, ksize=3, borderType=cv2.BORDER_REPLICATE)` (Gauss **yok**; isteyen önceden bulanıklaştırır).
2. `l2=True`: $m=d_x^2+d_y^2$ ve eşikler kare alınır; değilse $m=|d_x|+|d_y|$. Eşikler `floor` ile tamsayı.
3. NMS tamsayıyla: `TG22 = 13573` ($\tan22.5°\cdot2^{15}$), `ax = |dx|`, `ay = |dy| << 15`, `tg22x = ax*TG22`, `tg67x = tg22x + (ax << 16)`.
   - `ay < tg22x` → yatay: `m > sol` ve `m >= sağ`.
   - `ay > tg67x` → dikey: `m > üst` ve `m >= alt`.
   - yoksa çapraz: `s = -1 if (dx ^ dy) < 0 else 1`; `m > m[y-1, x-s]` ve `m > m[y+1, x+s]`.
   - Görüntünün dışındaki büyüklükler 0. Aday olmak için ayrıca `m > lo`.
4. Güçlü: aday ve `m > hi`. Sonuç: en az bir güçlü piksel içeren 8-bağlı aday bileşenleri, 255.

**`marr_hildreth(img, sigma, thr_frac=0.04)`** — $n$ = en küçük tek $\ge\lceil6\sigma\rceil$; `g = cv2.Laplacian(cv2.GaussianBlur(img.astype(np.float64), (n, n), sigma), cv2.CV_64F)`; b'deki sıfır geçişi kuralı, eşik `thr_frac * g.max()`, sınır 0. `bool` döndür.

In [ ]:
%%writefile stage_c.py
import numpy as np
import cv2
from scipy import ndimage


def canny_cv(img, lo, hi, l2=False):
    ...


def marr_hildreth(img, sigma, thr_frac=0.04):
    ...


from skimage import data
print((canny_cv(data.camera(), 50, 150) > 0).sum())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Canny'nin sonucu: gürültülü 1-B basamaklar için en iyi doğrusal dedektör ≈ Gauss türevi. Bunu öğrenerek görüyoruz.

- `dog_kernel(sigma, n)` — korelasyon (yani `conv1d`) çekirdeği olarak $k(x)=\frac{x}{\sigma^2}e^{-x^2/2\sigma^2}$, $x=-n/2,…,n/2$ (tamsayı).
- `learn_edge_filter(n=21, steps=1500, noise=0.3, seed=0)` — `make_batch` (hazır) gürültülü parça parça sabit sinyaller ve hedef olarak basamak noktalarında yumuşak tepeler verir. Sıfırdan başlayan uzunluğu `n` olan tek bir filtre `w`'yu `F.conv1d(x, w, padding=n//2)` çıkışının hedefe MSE'si (kenarlardan `n` örnek atılarak) ile Adam (lr = 1e-2), 64 × 256'lık yığınlarla öğren; `w`'yu döndür.
- `sobel_torch(img)` — `img (H, W)` float; **tek bir** `conv2d` (2 çıkış kanalı) ve `replicate` dolgusuyla `(gx, gy)`, `cv2.Sobel(…, cv2.CV_64F, …, borderType=cv2.BORDER_REPLICATE)` ile aynı.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F


def make_batch(B, L, g, noise, n):
    """Gürültülü parça parça sabit sinyaller ve yumuşak basamak hedefi."""
    jumps = (torch.rand(B, L, generator=g) < 0.03).float() * torch.randn(B, L, generator=g)
    x = jumps.cumsum(1) + noise * torch.randn(B, L, generator=g)
    t = torch.arange(n) - n // 2
    gauss = torch.exp(-t.float() ** 2 / 8) / (2 * (2 * torch.pi) ** 0.5)
    target = F.conv1d(jumps[:, None], gauss[None, None], padding=n // 2)[:, 0]
    return x, target


def dog_kernel(sigma, n):
    ...


def learn_edge_filter(n=21, steps=1500, noise=0.3, seed=0):
    ...


def sobel_torch(img):
    ...


print(dog_kernel(2.0, 7))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`canny_torch(imgs, lo, hi, l2=False)` — `imgs (B, H, W)` uint8 tensör → `(B, H, W)` uint8 (0/255), her görüntü için `cv2.Canny` ile **birebir** aynı.

- Sobel: c'deki gibi tek `conv2d`, replicate dolgu, `float` hesaplayıp `round().long()` ile tamsayıya geç (değerler küçük, kesin).
- NMS'i `int64` tensörlerle ve kaydırılmış görünümlerle (`F.pad` ile sıfır dolgu) yaz; bit kaydırma `<<` tensörlerde de çalışır.
- Histerezis: bileşen etiketleme yok; `edges ← aday ∧ maxpool3×3(edges)` adımını değişiklik kalmayana kadar tekrarla (tek `while` döngüsü serbest). Kaç tekrar sürdüğüne bak: en uzun kenar zincirinin uzunluğu kadar.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def canny_torch(imgs, lo, hi, l2=False):
    """imgs (B, H, W) uint8 → (B, H, W) uint8 (0/255)."""
    ...


imgs = torch.randint(0, 256, (2, 32, 32), dtype=torch.uint8)
print(canny_torch(imgs, 50, 150).shape)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Kirsch pusula çekirdekleri: 8 yön, en güçlü yanıt

`kirsch(img)` → `(resp, idx)`: `resp` 8 yanıtın en büyüğü (`float64`), `idx` en büyüğün yön indisi (eşitlikte küçük indis). Çekirdekler: K yönü $\begin{bmatrix}5&5&5\\-3&0&-3\\-3&-3&-3\end{bmatrix}$; dış halkayı sol üstten saat yönünde okursan `[5,5,5,-3,-3,-3,-3,-3]`. $i$. çekirdeğin halkası `np.roll(halka, -i)` (her adım 45° saat yönü tersine): 0 K, 1 KB, 2 B, 3 GB, 4 G, 5 GD, 6 D, 7 KD. Korelasyon: `cv2.filter2D(img.astype(np.float64), -1, k, borderType=cv2.BORDER_REPLICATE)`.

In [ ]:
%%writefile extra_x1.py
import numpy as np
import cv2


def kirsch(img):
    ...


from skimage import data
r, i = kirsch(data.camera())
print(r.max(), np.bincount(i.ravel(), minlength=8))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Hough dönüşümü: kenarlardan doğrulara

**Hough dönüşümü** (Gonzalez 10.2): her kenar pikseli $(x,y)$ parametre uzayında bir sinüs çizer, $\rho=x\cos\theta+y\sin\theta$; doğrular birikim dizisinin tepeleridir.

- `hough_lines(edges, thetas)` → `(acc, dists)`; `skimage.transform.hough_line` ile aynı: `off = ceil(hypot(H, W))`, satır indisi `round(ρ) + off` (`np.round`), `acc` boyutu `(2·off+1, len(thetas))`, `dists = linspace(-off, off, 2·off+1)`. Döngüsüz (`np.add.at`).
- `top_lines(acc, thetas, dists, k, nb=(10, 10))` → en güçlü `k` doğru `[(rho, theta), …]`: en büyüğü al, çevresindeki `±nb[0]` satır ve `±nb[1]` sütunu sıfırla, tekrarla.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def hough_lines(edges, thetas):
    ...


def top_lines(acc, thetas, dists, k, nb=(10, 10)):
    ...


thetas = np.linspace(-np.pi / 2, np.pi / 2, 180, endpoint=False)
e = np.zeros((50, 50), bool); e[25, :] = True
acc, d = hough_lines(e, thetas)
print(top_lines(acc, thetas, d, 1))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Gürültü türevleri ve LoG çekirdeğinin iki yolu

- `gaussian_size(sigma)` — en küçük tek $n\ge\lceil6\sigma\rceil$.
- `log_formula(sigma)` — $n\times n$ ızgarada LoG formülünü örnekle, ortalamasını çıkararak toplamı sıfıra getir.
- `log_via_laplacian(sigma)` — $n\times n$ Gauss çekirdeği (toplamı 1) ile Laplacian $\begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}$'nin **tam** (`full`) konvolüsyonu → $(n+2)\times(n+2)$; toplamı kendiliğinden 0.
- `derivative_noise(noise_var, smooth=None, seed=0)` — 256 örnekli rampa ($x<96$: 0, $96..160$: doğrusal 0 → 200, sonra 200) + $\mathcal N(0,\text{noise\_var})$; `smooth` verilirse `scipy.ndimage.gaussian_filter1d(…, smooth)`; düz bölgede (`[10:80]`) birinci ve ikinci farkların (`np.diff`) standart sapmalarını döndür.

In [ ]:
%%writefile extra_x3.py
import numpy as np
from scipy import ndimage, signal


def gaussian_size(sigma):
    ...


def log_formula(sigma):
    ...


def log_via_laplacian(sigma):
    ...


def derivative_noise(noise_var, smooth=None, seed=0):
    ...


print(gaussian_size(1.4), log_formula(1.4).sum())

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: HED: derin denetimli bütünsel kenar tespiti

[Xie & Tu — Holistically-Nested Edge Detection (ICCV 2015)](https://arxiv.org/abs/1504.06375)

VGG16 omurgasının beş aşamasından yan çıkışlar (side outputs) alan, her birine sınıf dengeli ikili çapraz entropi uygulayan ve öğrenilen ağırlıklarla birleştiren HED'i uygula; BSDS500 üzerinde eğitip ODS/OIS F-ölçüsünü Canny ile karşılaştır.


**Adımlar**

1. BSDS500'ü indir; çoklu insan etiketini ortalayıp eşiğe göre ikili etiket üret.
2. Yan çıkışlar: 1 × 1 konvolüsyon + transposed conv ile tam çözünürlüğe getir; sınıf dengeli kayıp $\beta=|Y_-|/|Y|$.
3. Birleşim katmanı (fuse) ve derin denetim (deep supervision) kayıplarını topla.
4. Değerlendirme: tahmini NMS ile incelt (bu modüldeki NMS'i yeniden kullan), toleranslı eşleştirmeyle ODS ve OIS hesapla.

**Kabul ölçütleri**

- BSDS500 testinde ODS ≥ 0.75 (makale 0.782); Canny satırı da tabloda.
- Derin denetim kapalıyken ve açıkken eğitim eğrileri karşılaştırılmış.
- Yan çıkışların görselleri: sığ katmanların ince, derin katmanların kaba kenarlar verdiği gösterilmiş.

## Kaynaklar

- [Gonzalez & Woods — Digital Image Processing, Bölüm 10.2](https://www.imageprocessingplace.com/) — ©
- [Canny — A Computational Approach to Edge Detection (IEEE PAMI 1986)](https://doi.org/10.1109/TPAMI.1986.4767851) — ©
- [Marr & Hildreth — Theory of Edge Detection (Proc. R. Soc. B 1980)](https://doi.org/10.1098/rspb.1980.0020) — ©
- [OpenCV — imgproc/src/canny.cpp](https://github.com/opencv/opencv/blob/4.x/modules/imgproc/src/canny.cpp) — Apache-2.0
- [scikit-image — hough_line](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.hough_line) — BSD-3